# Nairobi Flood Segmentation Model Training
## Google Colab GPU Training

This notebook trains the flood segmentation model on Google Colab's free GPU (T4 or P100).

**Time:** ~4 hours on GPU | **Cost:** Free | **Data:** 6.1 GB

## 1. Setup Environment & GPU

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi

## 2. Clone Repository

In [ ]:
import os
os.chdir('/content')

# Clone the repository
!git clone https://github.com/YOUR_USERNAME/nairobi-flood-digital-twi.git

os.chdir('nairobi-flood-digital-twi')
print("✓ Repository cloned")
!ls -la

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q numpy scipy scikit-learn tqdm matplotlib
!pip install -q python-dotenv

print("✓ All dependencies installed")

## 4. Prepare Training Data

In [ ]:
import os
import shutil

# Create directories
os.makedirs('data/processed/arrays', exist_ok=True)

# Path to dataset on Google Drive
# EDIT THIS: Replace with YOUR path to the dataset
DRIVE_DATASET_PATH = '/content/drive/MyDrive/nairobi-flood-data/segmentation_train_dataset.npz'
LOCAL_DATASET_PATH = 'data/processed/arrays/segmentation_train_dataset.npz'

# Copy dataset from Drive to Colab
if os.path.exists(DRIVE_DATASET_PATH):
    print(f"Found dataset, copying...")
    shutil.copy(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    file_size_gb = os.path.getsize(LOCAL_DATASET_PATH) / 1e9
    print(f"✓ Dataset copied ({file_size_gb:.1f} GB)")
else:
    print(f"⚠ Dataset NOT found at {DRIVE_DATASET_PATH}")
    print(f"Please upload segmentation_train_dataset.npz to Google Drive first")
    print(f"Expected path: My Drive > nairobi-flood-data > segmentation_train_dataset.npz")

# Verify
if os.path.exists(LOCAL_DATASET_PATH):
    import numpy as np
    data = np.load(LOCAL_DATASET_PATH, allow_pickle=False)
    print(f"\nDataset verified:")
    print(f"  X_train: {data['X_train'].shape}")
    print(f"  X_val:   {data['X_val'].shape}")
    print(f"  X_test:  {data['X_test'].shape}")

## 5. Train Model

In [ ]:
# Run training
# This takes ~4 hours on T4 GPU
import subprocess
import os

os.chdir('/content/nairobi-flood-digital-twi')

print("Starting training... (this will take ~4 hours on GPU)")
print("You can monitor progress in the output below.\n")

result = subprocess.run(
    ['python', '-m', 'src.models.train_segmentation'],
    capture_output=False
)

if result.returncode == 0:
    print("\n✓ Training completed successfully!")
else:
    print(f"\n✗ Training failed with exit code {result.returncode}")

## 6. Save Results to Google Drive

In [ ]:
import shutil
import os
import json

# Create output directory in Drive
output_dir = '/content/drive/MyDrive/nairobi-flood-data/training-outputs'
os.makedirs(output_dir, exist_ok=True)

# Copy trained model
model_src = 'models/time_series/segmentation_model.pth'
model_dst = f'{output_dir}/segmentation_model.pth'
if os.path.exists(model_src):
    shutil.copy(model_src, model_dst)
    print(f"✓ Model saved: {model_dst}")
else:
    print(f"✗ Model not found at {model_src}")

# Copy metrics
metrics_src = 'models/time_series/segmentation_metrics.json'
metrics_dst = f'{output_dir}/segmentation_metrics.json'
if os.path.exists(metrics_src):
    shutil.copy(metrics_src, metrics_dst)
    print(f"✓ Metrics saved: {metrics_dst}")
    
    # Display metrics summary
    with open(metrics_src, 'r') as f:
        metrics = json.load(f)
        if 'test_metrics' in metrics:
            test = metrics['test_metrics']
            print(f"\nTest Results:")
            print(f"  IoU:       {test['iou']:.4f}")
            print(f"  F1:        {test['f1']:.4f}")
            print(f"  Precision: {test['precision']:.4f}")
            print(f"  Recall:    {test['recall']:.4f}")
else:
    print(f"✗ Metrics not found at {metrics_src}")

print(f"\n✓ All outputs saved to: {output_dir}")

## 7. Download Model to Your Laptop

In [ ]:
# Instructions for downloading
print("""
✓ Training Complete!

Next steps:

1. Download model from Google Drive:
   - Go to: My Drive > nairobi-flood-data > training-outputs
   - Download: segmentation_model.pth
   - Download: segmentation_metrics.json
   - Move to: models/time_series/ on your laptop

2. Or use wget:
   gdown <FILE_ID> -O models/time_series/segmentation_model.pth

3. Then validate locally:
   python -c "import torch; model = torch.load('models/time_series/segmentation_model.pth')"

4. Use in predictions (see training_notes.md)
""")